# Dev Pipeline Benchmark
Notebook này kiểm thử toàn bộ luồng RAG từ Crawl -> Chunk -> Embed -> Retrieve -> Evaluate trên tập Dev 50 samples.

In [ ]:
import pandas as pd
dev_qrels = pd.read_csv('../data/processed/dev_qrels_synthetic_groq.csv')
display(dev_qrels.head())

# Extract unique URLs to crawl
urls_to_crawl = dev_qrels['url'].unique().tolist()
print(f"Tổng số URLs cần crawl: {len(urls_to_crawl)}")

## 1. Web Scraper (aiohttp + trafilatura)
Sử dụng luồng bất đồng bộ để tải văn bản nhanh chóng, loại bỏ HTML thừa thãi.

In [ ]:
import asyncio
import aiohttp
import trafilatura
from tqdm.asyncio import tqdm

async def fetch_and_extract(session, url):
    try:
        async with session.get(url, timeout=10) as response:
            if response.status != 200:
                return url, None
            html = await response.text()
            # trafilatura extracts main text, removes ads/menus
            text = trafilatura.extract(html)
            return url, text
    except Exception as e:
        return url, None

async def crawl_urls(urls):
    async with aiohttp.ClientSession() as session:
        tasks = [fetch_and_extract(session, u) for u in urls]
        # Run concurrently with progress bar
        results = await tqdm.gather(*tasks)
    return results

# LƯU Ý: Trong Jupyter Notebook, event loop đã chạy sẵn, 
# nên dùng await trực tiếp thay vì asyncio.run()
scraped_data = await crawl_urls(urls_to_crawl)

# Lọc bỏ các URL lỗi (None)
valid_docs = [{'url': u, 'content': t} for u, t in scraped_data if t is not None]
df_docs = pd.DataFrame(valid_docs)
print(f"Thu thập thành công {len(df_docs)} / {len(urls_to_crawl)} bài báo.")
display(df_docs.head())